In [1]:
import os
import asyncio
from google.adk.sessions import InMemorySessionService
from google.adk.artifacts import InMemoryArtifactService
from google.adk.runners import Runner
from google.adk.agents import LlmAgent
import google.genai.types as types
from config import config

# -----------------------------
# Setup API Key and model
# -----------------------------
APP_NAME = "chatbot_demo"
USER_ID = "user_001"
SESSION_ID = "session_001"
MODEL = "gemini-2.5-flash"

os.environ["GOOGLE_API_KEY"] = config.GOOGLE_API_KEY.strip()

# -----------------------------
# Create LlmAgent (chatbot)
# -----------------------------
root_agent = LlmAgent(
    model=MODEL,
    name="ChatAgent",
    instruction="""
    You are a helpful assistant. Respond politely and concisely to user questions.
    """
)

# -----------------------------
# Setup In-Memory Services
# -----------------------------
session_service = InMemorySessionService()
artifact_service = InMemoryArtifactService()

# Create a new session
async def create_session():
    await session_service.create_session(
        app_name=APP_NAME,
        user_id=USER_ID,
        session_id=SESSION_ID
    )

# -----------------------------
# Setup Runner
# -----------------------------
runner = Runner(
    agent=root_agent,
    app_name=APP_NAME,
    session_service=session_service,
    artifact_service=artifact_service
)

# -----------------------------
# Chat function
# -----------------------------
async def chat(user_input):
    content = types.Content(
        role="user",
        parts=[types.Part(text=user_input)]
    )
    
    events = runner.run_async(
        user_id=USER_ID,
        session_id=SESSION_ID,
        new_message=content
    )
    
    async for event in events:
        if event.is_final_response() and event.content and event.content.parts:
            text = event.content.parts[0].text
            print("ChatAgent:", text)
            return text
    return None

# -----------------------------
# Video Script Generation using raw content
# -----------------------------
async def generate_video_script_from_file(input_filename):
    # Read raw content from file
    try:
        with open(input_filename, "r", encoding="utf-8") as f:
            raw_content = f.read()
    except FileNotFoundError:
        print(f"File '{input_filename}' not found.")
        return
    
    prompt = f"""
    You are an AI Engineer with 1+ year of experience.
    Using the following raw content, create a professional video script.
    Make it simple, clear, and suitable for 5-10 minutes of video content.
    Include introduction, key points, and conclusion.
    There could be multiple Developers work give you need to make one script including all content.

    Raw content:
    {raw_content}
    """
    
    script_content = await chat(prompt)
    
    if script_content:
        output_filename = "video_script.txt"
        with open(output_filename, "w", encoding="utf-8") as f:
            f.write(script_content)
        print(f"Video script saved to {output_filename}")
    else:
        print("Failed to generate video script.")

# -----------------------------
# Main execution
# -----------------------------
async def main():
    await create_session()
    
    # Example chat
    await chat("Hello! Can you suggest a fun weekend activity?")
    
    # Ask user for filename
    input_filename = input("Enter the filename containing your raw content: ").strip()
    await generate_video_script_from_file(input_filename)

# if __name__ == "__main__":
#     asyncio.run(main())

await main()


ChatAgent: Hello! How about a picnic in a park?


Enter the filename containing your raw content:  script.txt


ChatAgent: Okay, here's a professional video script, drawing from both contributions, suitable for a 5-10 minute presentation.

---

**Video Script: Building Advanced Tool-Aware AI Assistants for the Enterprise**

**Target Audience:** Developers, AI Enthusiasts, Enterprise IT Managers
**Video Length:** ~7-9 minutes

---

**(Intro Music Fades In, then out as Host speaks)**

**[SCENE 1: INTRODUCTION]**

**HOST (AI Engineer, professional attire, engaging tone)**
"Hello everyone, and welcome! As AI Engineers, we're constantly looking for ways to bridge the gap between complex enterprise tools and intuitive user interaction. In today's fast-paced environments, organizations rely on a multitude of applications – from project management in Jira, to documentation in Confluence, to essential communication in Gmail. The challenge? Making these tools work together seamlessly, without manual effort or convoluted processes."

"Today, we're going to dive into how we're building the next generation o